# Dati reali DR2low analizzati con scattering α = 4.4

Stessi dati reali di J1640+2224 (DR2low) del notebook 01, ma la likelihood modella lo scattering con **α = 4.4** (valore atteso per una turbolenza di Kolmogorov) invece del solito 4.0. Il modello resta a 6 parametri: α è fisso, non libero.

Confronto con `chain` (stessi dati, α = 4.0).

**Attenzione.** Sui dati reali non esistono valori veri, quindi qui non si parla di bias: si misura solo quanto le stime dipendono dall'indice cromatico assunto. Dalle simulazioni (notebook 07 e 09) sappiamo cosa aspettarci: `sv_gp_log10_A` si sposta di circa Δα·log₁₀(1400/ν_eff) ≈ 0.4 dex, gli altri parametri quasi nulla. Se sui dati reali il comportamento fosse diverso, vorrebbe dire che lo scattering reale non si comporta come nelle simulazioni.

## Setup comune

Copia delle celle 0, 1, 3, 4, 6, 7, 9 di `Likelihood.ipynb`: il modello standard con α = 4.0, che qui serve da riferimento.

In [1]:
import numpy as np
import os, glob, json
import pickle

import sys

# this is the path to the discovery folder
sys.path.insert(0, "./")
import discovery_new as ds

import matplotlib.pyplot as plt
import scipy
import jax.numpy as jnp

from scipy.stats import uniform
import re
from PTMCMCSampler.PTMCMCSampler import PTSampler as ptmcmc
from scipy.stats import uniform
import jax

import time
import emcee

Optional mpi4py package is not installed.  MPI support is not available.


In [2]:
# path to the folder with the data
path = './dati/DR2low_feathers/'
s_dsfiles = np.sort(os.listdir(path))
psr = [ds.pulsar.Pulsar.read_feather(path + f'{psrfile}') for psrfile in s_dsfiles][9] # we are only analysing pulsar #9

In [3]:
# data
model = [psr.residuals]
# timing model
model += [ds.makegp_timing(psr, svd=True)]
# white noise parameters
model += [ds.makenoise_measurement(psr, psr.noisedict, tnequad = True, ecorr=True)]
# red noise paramters
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_red_components'], T=ds.getspan(psr), name='red_noise' ))
# DM variation parameters
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_dm_components'], T=ds.getspan(psr), name='dm_gp', fourierbasis=ds.make_dmfourierbasis(alpha = 2.0, tndm = False)))
# scattering noise parameters
model.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_chrom_components'], T=ds.getspan(psr), name='sv_gp', fourierbasis=ds.make_dmfourierbasis(alpha = 4.0, tndm = False)))

In [4]:
logL = ds.PulsarLikelihood(model).logL

In [5]:
# annoying function needed because the function logL want a dictionary as input, but our sampler wants a likelihood with an array as input
# so we have to switch from one to another :(
def params_dict_from_array(x):
    # ensure x is a plain numpy array before slicing
    x = np.asarray(x)
    return dict(zip(logL.params, x))

In [6]:
##### Cell for the prior distributions ###################################################################################

# dictionary with minimum and maximum value of the prior distribution
my_priordict = {}
my_priordict["(.*_)?sv_gp_gamma.*"] = [0, 7]
my_priordict["(.*_)?sv_gp_alpha.*"] = [0, 10]
my_priordict["(.*_)?sv_gp_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?sw_log10_A.*"] = [-10, 1]
my_priordict["(.*_)?sw_gamma.*"] = [-5, 6]
my_priordict["(.*_)?dm_gp_gamma.*"] = [0, 7]
my_priordict["(.*_)?dm_gp_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?red_noise_gamma.*"] = [0, 7]
my_priordict["(.*_)?red_noise_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?sw_n_earth.*"] = [0, 30]


def build_priors(param_names, priordict):
    """
    Build a {param_name: scipy dist} dict with uniform prior distributions in the range
    """
    priors = {}
    unmatched = []
    for name in param_names:
        matched = False
        for pattern, (lo, hi) in priordict.items():
            if re.fullmatch(pattern, name):
                priors[name] = uniform(loc=lo, scale=hi - lo) ## this is where the uniform distribution is defined
                matched = True
                break
        if not matched:
            unmatched.append(name)

    if unmatched:
        print(f"Warning: no prior found for {len(unmatched)} parameters, "
              f"falling back to standard normal:\n  {unmatched}")
    return priors

priors = build_priors(logL.params, my_priordict)
priors

def log_prior(x):
    """Compute log prior"""
    params = params_dict_from_array(x)
    
    logp = 0.0
    for name, value in params.items():
        if name in priors:
            logp += priors[name].logpdf(value)
        else:
            # Default: standard normal prior
            if isinstance(value, np.ndarray):
                logp += -0.5 * np.sum(value**2)
            else:
                logp += -0.5 * value**2
    
    return logp

def prior_draw():
    """Draw from prior and return as flat array"""
    x = []
    for name in logL.params:
        x.append(priors[name].rvs())
    return np.array(x)

x0 = prior_draw()
x0

array([  4.40183572, -13.4017294 ,   2.61914355, -16.1723374 ,
         3.2161853 , -16.042264  ])

In [7]:
@jax.jit # compile the likelihood in jax to make it faster
def _jit_logL(params_array):
    params = dict(zip(logL.params, params_array))
    return logL(params)

_jit_logL(x0)

def log_likelihood(x): 
    """
    Likelihood function that is given to the samples, it uses the jitted (=faster) version of the likelihood
    """
    x = np.asarray(x, dtype=np.float64)
    result = _jit_logL(jnp.array(x))
    return float(np.array(result))  # always a plain Python float

## Catena di riferimento: dati reali con α = 4.0 (cella 13 dell'originale)

In [8]:
# read the output chain that has shape (n_samples, n_parameters)
chain = np.loadtxt('./test/chain_1.txt')[:, :-4]
chain = chain[len(chain) // 4:] # remove first 25% from the chain
chain.shape

(75001, 6)

## Modello sui dati reali con α = 4.4

In [9]:
# identico al modello standard, tranne alpha nella riga sv_gp: 4.4 invece di 4.0
model_real_sv44 = [psr.residuals]
model_real_sv44 += [ds.makegp_timing(psr, svd=True)]
model_real_sv44 += [ds.makenoise_measurement(psr, psr.noisedict, tnequad=True, ecorr=True)]
model_real_sv44.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_red_components'],
                                         T=ds.getspan(psr), name='red_noise'))
model_real_sv44.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_dm_components'],
                                         T=ds.getspan(psr), name='dm_gp',
                                         fourierbasis=ds.make_dmfourierbasis(alpha=2.0, tndm=False)))
model_real_sv44.append(ds.makegp_fourier(psr, ds.powerlaw, psr.noisedict[psr.name + '_chrom_components'],
                                         T=ds.getspan(psr), name='sv_gp',
                                         fourierbasis=ds.make_dmfourierbasis(alpha=4.4, tndm=False)))   # <-- 4.4

logL_real_sv44 = ds.PulsarLikelihood(model_real_sv44).logL
logL_real_sv44.params

['J1640+2224_dm_gp_gamma',
 'J1640+2224_dm_gp_log10_A',
 'J1640+2224_red_noise_gamma',
 'J1640+2224_red_noise_log10_A',
 'J1640+2224_sv_gp_gamma',
 'J1640+2224_sv_gp_log10_A']

In [10]:
# controllo: stessi parametri e stesso ordine di logL.params (alpha e' fisso, non e' un parametro)
assert list(logL_real_sv44.params) == list(logL.params), (logL_real_sv44.params, logL.params)
print('OK: 6 parametri, identici a logL.params')

OK: 6 parametri, identici a logL.params


In [11]:
def params_dict_from_array_real_sv44(x):
    x = np.asarray(x)
    return dict(zip(logL_real_sv44.params, x))

priors_real_sv44 = build_priors(logL_real_sv44.params, my_priordict)

def log_prior_real_sv44(x):
    params = params_dict_from_array_real_sv44(x)
    logp = 0.0
    for name, value in params.items():
        if name in priors_real_sv44:
            logp += priors_real_sv44[name].logpdf(value)
        else:
            logp += -0.5 * np.sum(np.asarray(value)**2)
    return logp

def prior_draw_real_sv44():
    return np.array([priors_real_sv44[name].rvs() for name in logL_real_sv44.params])

x0_real_sv44 = prior_draw_real_sv44()

In [12]:
@jax.jit
def _jit_logL_real_sv44(params_array):
    return logL_real_sv44(dict(zip(logL_real_sv44.params, params_array)))

def log_likelihood_real_sv44(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.array(_jit_logL_real_sv44(jnp.array(x))))

print('logL iniziale =', log_likelihood_real_sv44(x0_real_sv44))

start = time.time()
for _ in range(20):
    log_likelihood_real_sv44(x0_real_sv44)
dt = (time.time() - start) / 20
print(f'una valutazione = {dt*1e3:.1f} ms  ->  10^6 iterazioni ~ {dt*1e6/3600:.1f} ore (solo likelihood)')

logL iniziale = 67902.98947288068
una valutazione = 1.0 ms  ->  10^6 iterazioni ~ 0.3 ore (solo likelihood)


In [13]:
sampler_real_sv44 = ptmcmc(
    ndim=len(logL_real_sv44.params),
    logl=log_likelihood_real_sv44,
    logp=log_prior_real_sv44,
    cov=np.eye(len(logL_real_sv44.params)) * 0.001,
    outDir='./test_real_sv44/',
    resume=True
)

### Sampling

Eseguire **una sola volta**. Con α fisso il costo per passo è quello solito e τ è piccolo (~6): 10⁶ iterazioni danno ESS ≈ 12000, molto più del necessario. Per confrontabilità con la run a α = 4.0 restano 10⁶; 2×10⁵ basterebbero (ESS ≈ 2000).

In [14]:
print("Starting sampling (dati REALI DR2low, scattering con alpha=4.4)...")
sampler_real_sv44.sample(x0_real_sv44, Niter=1000000, thin=10)

Starting sampling (dati REALI DR2low, scattering con alpha=4.4)...
Finished 1.00 percent (1.00 percent of new work) in 13.156553 s Acceptance rate = 0.2288Adding DE jump with weight 20
Finished 100.00 percent (100.00 percent of new work) in 1300.614177 s Acceptance rate = 0.169649
Run Complete


## Catena e risultati

In [15]:
chain_real_sv44 = np.loadtxt('./test_real_sv44/chain_1.txt')[:, :-4]
chain_real_sv44 = chain_real_sv44[len(chain_real_sv44)//4:]   # rimuovo il 25% di burn-in
chain_real_sv44.shape

(75001, 6)

In [ ]:
import emcee
tau_real_sv44 = emcee.autocorr.integrated_time(chain_real_sv44, has_walkers=False, quiet=True)
print('tau =', np.round(tau_real_sv44, 1))
print('Effective length =', int(len(chain_real_sv44) / tau_real_sv44.max()))

### Spostamento dei parametri e confronto con la previsione

Lo spostamento atteso per `sv_gp_log10_A` passando da α = 4.0 a α = 4.4 nell'analisi è **−Δα·log₁₀(1400/ν_eff)**: il segno è negativo perché qui è il modello, non i dati, ad avere l'esponente più ripido. ν_eff è stimata pesando i TOA con il loro contributo al segnale di scattering.

In [ ]:
# frequenza efficace dei TOA reali e spostamento previsto
f_toa = np.asarray(psr.freqs)
w = (1400.0 / f_toa)**(2 * 4.4) / np.asarray(psr.toaerrs)**2
lever = np.sum(w * np.log10(1400.0 / f_toa)) / np.sum(w)
nu_eff = 1400.0 / 10**lever
shift_atteso = -0.4 * lever
print(f'nu_eff ~ {nu_eff:.0f} MHz, spostamento atteso su sv_gp_log10_A = {shift_atteso:+.2f} dex')

print()
print(f"{'parametro':20s} | {'alpha=4.0':>22s} | {'alpha=4.4':>22s} | scostamento/sigma")
for i, name in enumerate(logL.params):
    a16, a50, a84 = np.percentile(chain[:, i], [16, 50, 84])
    b16, b50, b84 = np.percentile(chain_real_sv44[:, i], [16, 50, 84])
    s = np.sqrt(((a84 - a16)/2)**2 + ((b84 - b16)/2)**2)
    print(f"{name.replace('J1640+2224_', ''):20s} | {a50:8.3f} [{a16:6.2f},{a84:6.2f}] | {b50:8.3f} [{b16:6.2f},{b84:6.2f}] | {(b50 - a50)/s:6.2f}")

print()
print('spostamento osservato su sv_gp_log10_A = '
      f"{np.median(chain_real_sv44[:, list(logL.params).index('J1640+2224_sv_gp_log10_A')]) - np.median(chain[:, list(logL.params).index('J1640+2224_sv_gp_log10_A')]):+.2f} dex")

### Quale dei due modelli descrive meglio i dati reali?

Le ultime colonne della catena di PTMCMC contengono log-likelihood e log-posterior di ogni campione. Confrontare massimo e mediana della log-likelihood dei due run dà un'indicazione: i due modelli hanno lo stesso numero di parametri e le stesse prior, quindi un valore sistematicamente più alto favorisce quell'indice cromatico.

**Non è un fattore di Bayes**: sarebbe necessaria l'evidenza, cioè la likelihood marginalizzata, che PTMCMC non calcola. Differenze di ΔlogL di pochi decimi non sono significative; qualche unità lo è.

In [ ]:
def logl_col(path):
    c = np.loadtxt(path)
    c = c[len(c)//4:]
    return c[:, -3]   # colonna della log-likelihood

ll_40 = logl_col('./test/chain_1.txt')
ll_44 = logl_col('./test_real_sv44/chain_1.txt')

print(f"alpha = 4.0 : logL max {ll_40.max():.2f}, mediana {np.median(ll_40):.2f}")
print(f"alpha = 4.4 : logL max {ll_44.max():.2f}, mediana {np.median(ll_44):.2f}")
print(f"Delta logL (4.4 - 4.0) : max {ll_44.max() - ll_40.max():+.2f}, mediana {np.median(ll_44) - np.median(ll_40):+.2f}")

### Corner di confronto

In [ ]:
import corner

clean_labels = [name.replace('J1640+2224_', '') for name in logL.params]

# dati reali, alpha = 4.0 — blu
fig = corner.corner(
    chain,
    labels=clean_labels,
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#2E86AB',
)

# dati reali, alpha = 4.4 — blu medio
corner.corner(
    chain_real_sv44,
    labels=clean_labels,
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 11},
    hist_kwargs={'density': True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    bins=40,
    color='#457B9D',
    fig=fig
)

fig.set_size_inches(16, 16)
fig.suptitle('J1640+2224 DR2low reali — scattering modellato con α=4.0 (blu) vs α=4.4 (blu medio)', fontsize=14, y=1.00)
fig.savefig('corner_reali_alpha40_vs_alpha44.png', dpi=150, bbox_inches='tight')

## Riepilogo

_Da compilare dopo il run._

- Spostamento osservato su `sv_gp_log10_A`: … dex, contro il … dex previsto dalla frequenza efficace.
- Altri parametri: …
- ΔlogL tra i due modelli: …
- Conclusione: quanto le stime di rumore di J1640+2224 dipendono dall'indice cromatico assunto per lo scattering.